In [1]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
  try:
    return ast.parse(code_string)
  except SyntaxError:
    return None # some code snippets are not valid (python 2 instead of python 3)

In [2]:
### Linearizing the AST (by passing first node) into a list of tokens
def linearize_ast(node, tokens):
  if node is None:
    return

  tokens.append(type(node).__name__)

  # specific node types
  # if it's a name we add the variable name
  #if isinstance(node, ast.Name):
  #  tokens.append(f"VAR_{node.id}")

  # if it's a constant we add a placeholder to not write actual values
  #elif isinstance(node, ast.Constant):
  #  tokens.append("CONST")

  # if it's a function argument we add the argument name
  #elif isinstance(node, ast.arg):
  #  tokens.append(f"ARG_{node.arg}")

  for child in ast.iter_child_nodes(node):
    linearize_ast(child, tokens)

In [3]:
linearized_tree = []
linearize_ast(code_to_ast("def add(a, b): return a + b + 3"), linearized_tree)
print(linearized_tree)

['Module', 'FunctionDef', 'arguments', 'arg', 'arg', 'Return', 'BinOp', 'BinOp', 'Name', 'Load', 'Add', 'Name', 'Load', 'Add', 'Constant']


Let's now build a dictionary with Corpora, exactly as we have done before.

In [4]:
from datasets import load_dataset

train_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/train.jsonl",
  split = "train")
valid_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/valid.jsonl",
  split = "train")
test_dataset = load_dataset(
  "json",
  data_files = "./../../data/raw/dataset/python/test.jsonl",
  split = "train")

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
train_linearized_trees = []
for code in train_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  train_linearized_trees.append(linearized_tree)

valid_linearized_trees = []
for code in valid_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  valid_linearized_trees.append(linearized_tree)

test_linearized_trees = []
for code in test_dataset['code']:
  linearized_tree = []
  linearize_ast(code_to_ast(code), linearized_tree)
  test_linearized_trees.append(linearized_tree)

print(train_linearized_trees[0])

['Module', 'FunctionDef', 'arguments', 'arg', 'arg', 'Constant', 'Expr', 'Constant', 'Assign', 'Name', 'Store', 'BinOp', 'Name', 'Load', 'Add', 'Constant', 'Assign', 'Name', 'Store', 'Call', 'Attribute', 'Name', 'Load', 'Load', 'Name', 'Load', 'Return', 'BinOp', 'BinOp', 'Subscript', 'Name', 'Load', 'Constant', 'Load', 'Add', 'Name', 'Load', 'Add', 'Subscript', 'Call', 'Attribute', 'Subscript', 'Name', 'Load', 'Constant', 'Load', 'Load', 'Constant', 'Constant', 'Load']


In [6]:
from gensim import corpora
code_dictionary = corpora.Dictionary(train_linearized_trees)
special_tokens = {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}
code_dictionary.patch_with_special_tokens(special_tokens)

In [7]:
code_dictionary.token2id

{'Add': 91,
 'Assign': 92,
 'Attribute': 93,
 'BinOp': 94,
 'Call': 4,
 'Constant': 5,
 'Expr': 6,
 'FunctionDef': 7,
 'Load': 8,
 'Module': 9,
 'Name': 10,
 'Return': 11,
 'Store': 12,
 'Subscript': 13,
 'arg': 14,
 'arguments': 15,
 'Compare': 16,
 'Eq': 17,
 'ExceptHandler': 18,
 'If': 19,
 'Not': 20,
 'Try': 21,
 'UnaryOp': 22,
 'Raise': 23,
 'And': 24,
 'Assert': 25,
 'BoolOp': 26,
 'Continue': 27,
 'Dict': 28,
 'For': 29,
 'Gt': 30,
 'In': 31,
 'Is': 32,
 'ListComp': 33,
 'NotIn': 34,
 'Tuple': 35,
 'comprehension': 36,
 'List': 37,
 'With': 38,
 'withitem': 39,
 'Break': 40,
 'Slice': 41,
 'USub': 42,
 'Mult': 43,
 'IsNot': 44,
 'AugAssign': 45,
 'Div': 46,
 'GtE': 47,
 'Lt': 48,
 'LtE': 49,
 'Mod': 50,
 'Or': 51,
 'Sub': 52,
 'While': 53,
 'Yield': 54,
 'Pass': 55,
 'keyword': 56,
 'Lambda': 57,
 'DictComp': 58,
 'IfExp': 59,
 'NotEq': 60,
 'Starred': 61,
 'GeneratorExp': 62,
 'BitAnd': 63,
 'BitOr': 64,
 'BitXor': 65,
 'RShift': 66,
 'LShift': 67,
 'Del': 68,
 'Delete': 69,
 '

In [8]:
len(code_dictionary)

95

We can see that we went from a code_dictionary of over 1 million of tokens (see notebook 02) to just over 400k. This can help speed up the training and give some more accurate results. So, let's try this by just copy-pasting the model from notebook 03. We're gonna use the same vocab for docstrings. We're gonna save the processed dataset and test it on the next notebook.

In [9]:
from datasets import load_from_disk
old_dataset = load_from_disk("../../data/processed/tokenized_codexglue")

In [10]:
train_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in train_linearized_trees
]

valid_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in valid_linearized_trees
]

test_input_ids = [
  [code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
  for tree in test_linearized_trees
]

In [11]:
from datasets import Dataset, DatasetDict

processed_datasets = DatasetDict({
  'train': Dataset.from_dict({
    'input_ids': train_input_ids,
    'labels': old_dataset['train']['labels']
  }),
  'valid': Dataset.from_dict({
    'input_ids': valid_input_ids,
    'labels': old_dataset['valid']['labels']
  }),
  'test': Dataset.from_dict({
    'input_ids': test_input_ids,
    'labels': old_dataset['test']['labels']
  })
})

We're gonna save a copy of our docstring dictionary, so as to have everything organized.

In [12]:
docstring_dictionary = corpora.Dictionary.load('./../../data/processed/tokenized_codexglue/docstring_dictionary.pt')

In [13]:
processed_datasets.save_to_disk('./../../data/processed/ast01/')
code_dictionary.save('./../../data/processed/ast01/code_dictionary.pt')
docstring_dictionary.save('./../../data/processed/ast01/docstring_dictionary.pt')

Saving the dataset (1/1 shards): 100%|██████████| 14918/14918 [00:00<00:00, 1062987.40 examples/s]
